In [1]:
import sksurv
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
import pandas as pd
import numpy as np
import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
import os
import kagglehub
from dataclasses import asdict
from sksurv.util import Surv
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

from Utils.Config import Config, TrialResult
from Utils.ensemble_utils import collect_top_trial_oofs_from_configs
from Utils.utils import (
    set_seed,
    make_surv_y,
    create_features,
    HORIZONS
)
from Utils.ensemble_utils import find_ensemble_model
from Optuna_Experiment import SEEDS, MODEL_TYPES


Python:  3.11.14
OS:  Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39
Scikit-learn:  1.8.0
Scikit-survival:  0.27.0
Metadata path:  /home/user/.cache/kagglehub/competitions/WiDSWorldWide_GlobalDathon26/metaData.csv
Train path:  /home/user/.cache/kagglehub/competitions/WiDSWorldWide_GlobalDathon26/train.csv
Test path:  /home/user/.cache/kagglehub/competitions/WiDSWorldWide_GlobalDathon26/test.csv



In [2]:
COMP_DIR = kagglehub.competition_download('WiDSWorldWide_GlobalDathon26')
metadata_path = os.path.join(COMP_DIR, 'metaData.csv')
train_path = os.path.join(COMP_DIR, 'train.csv')
test_path = os.path.join(COMP_DIR, 'test.csv')

In [3]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
train_processed = create_features(train_df)
test_processed = create_features(test_df)

y_full = make_surv_y(
    event=train_processed['event'],
    time=train_processed['time_to_hit_hours']
)

event_horizon = np.array(HORIZONS).copy()
event_horizon[-1] = min(event_horizon[-1], train_processed['time_to_hit_hours'].max() - 1e-6)

print("Event horizons:", event_horizon)

Event horizons: [12.         24.         48.         66.99447313]


# Model Ensemble (differnt Config)

In [4]:
SEEDS = [42,]
MODEL_TYPES = ['coxnet', 'gbsa', 'rsf']

print("SEEDS: ", SEEDS)
print("MODEL TYPES: ", MODEL_TYPES)
use_norm = True
normalize_method = "rank"

SEEDS:  [42]
MODEL TYPES:  ['coxnet', 'gbsa', 'rsf']


In [5]:
top_ratios = {}
for model_type in MODEL_TYPES:
    top_ratios[model_type] = 0.5

full_oof_result = collect_top_trial_oofs_from_configs(
    SEEDS, MODEL_TYPES, train_processed,
    HORIZONS, top_ratios=top_ratios
)
print("Final Dict length: ", len(full_oof_result))

Saving OOF predictions for top 150 trials to TOP_OOF/42/coxnet...
Best trial value: 0.9642943769968599
Best Tral Number: 247
[collect] seed=42, model=coxnet, added=150, total=150

Saving OOF predictions for top 150 trials to TOP_OOF/42/gbsa...
Best trial value: 0.9629674137093398
Best Tral Number: 106
[collect] seed=42, model=gbsa, added=150, total=300

Saving OOF predictions for top 150 trials to TOP_OOF/42/rsf...
Best trial value: 0.96661274268073
Best Tral Number: 158
[collect] seed=42, model=rsf, added=150, total=450

Final Dict length:  450


In [6]:
# Risk Score Normalize
from scipy.stats import rankdata

def rank_normalize_risk(x):
    x = np.asarray(x, dtype=float)
    r = rankdata(x, method="average")
    return (r - 1) / (len(r) - 1 + 1e-8)

def normalize_risk(x, eps=1e-8):
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / (x.std() + eps)

if normalize_method.lower() == "z_score":
    normalize_fn = normalize_risk
elif normalize_method.lower() == "rank":
    normalize_fn = rank_normalize_risk
else:
    raise ValueError(f"Unknown normalize_method: {normalize_method}")

if use_norm:

    for key, item in full_oof_result.items():
        risk = item["oof_risk"]
        
        if not np.isfinite(risk).all():
            raise ValueError(f"NaN or Inf in risk_pred: {key}")
        
        full_oof_result[key]["oof_risk"] = normalize_fn(risk)
        
    print(f"Risk Score Normalize Complelte Method: {normalize_method}")


Risk Score Normalize Complelte Method: rank


# Select Ensemble Model

- 대회에 쓰이는 metric을 직접 최적화 하는 방식으로 진행

## 제약조건

1. 전체 trial중 30% (90개)
2. diversity를 어느정도 갖춘 core model [106,152,236,243]으로 시작
3. 이미 선택된 모델들과 corr이 0.995 이상으로 강하다면 skip
4. improve가 너무 적게 상승한다면 noise로 보고 skip
5. max model은 10개 정도 진행

## Config

In [7]:
COMP_DIR = kagglehub.competition_download('WiDSWorldWide_GlobalDathon26')
metadata_path = os.path.join(COMP_DIR, 'metaData.csv')
train_path = os.path.join(COMP_DIR, 'train.csv')
test_path = os.path.join(COMP_DIR, 'test.csv')
seed = 42
set_seed(seed)

train_df = pd.read_csv(train_path)
train_processed = create_features(train_df)

print("Metadata path: ", metadata_path)
print("Train path: ", train_path)
print("Test path: ", test_path)
print()

y_full = make_surv_y(
    event=train_processed['event'],
    time=train_processed['time_to_hit_hours']
)

Metadata path:  /home/user/.cache/kagglehub/competitions/WiDSWorldWide_GlobalDathon26/metaData.csv
Train path:  /home/user/.cache/kagglehub/competitions/WiDSWorldWide_GlobalDathon26/train.csv
Test path:  /home/user/.cache/kagglehub/competitions/WiDSWorldWide_GlobalDathon26/test.csv



In [12]:
# init_model_list = [106,152,236,243]
best_key, best_item = max(
    full_oof_result.items(),
    key=lambda x: x[1]["value"]
)

eps = 1e-5
init_model_list = [best_key]
max_pair_corr = 1. - eps
max_ensemble_corr = 1. - 2*eps
min_improvement_score = eps
max_model_num = 10
allow_duplicate = True
use_weight_grid_search = True

In [13]:
# 다른 모델도 후보로 넣으려면 
# .update 이용하여 딕셔너리 추가

# TODO 앙상블 모델당 weight도 추가로 grid search할수 있게 변경
# full_oof_result["oof_risk"] is already normalized before ensemble search
# Risk Score는 대쇠교가 중요해서 Denorm 할 필요가 없음

ensemble_result = find_ensemble_model(
    oof_result=full_oof_result,
    label=y_full,
    max_ensemble_corr=max_ensemble_corr,
    max_pair_corr=max_pair_corr,
    min_imporvement_score=min_improvement_score,
    max_model_num=max_model_num,
    init_model_list=init_model_list,
    horizons=HORIZONS,
    verbose=False,
    allow_duplicate=allow_duplicate,
    use_weight_grid_search=use_weight_grid_search
)

===== Initial Ensemble =====
Initial models: [(42, 'rsf', 158)]
Initial hybrid score: 0.967232

>>> SELECTED MODEL [2 / 10]
Trial: (42, 'rsf', 229)
Improvement: +0.000545
New hybrid score: 0.967777

>>> SELECTED MODEL [3 / 10]
Trial: (42, 'gbsa', 236)
Improvement: +0.000530
New hybrid score: 0.968307

>>> SELECTED MODEL [4 / 10]
Trial: (42, 'rsf', 175)
Improvement: +0.000031
New hybrid score: 0.968338

>>> SELECTED MODEL [5 / 10]
Trial: (42, 'rsf', 86)
Improvement: +0.000022
New hybrid score: 0.968360

No candidate model satisfies the conditions.
Max Pair Corr: 0.99999
Max Ensemble Corr: 0.99998
Min Improve Score: 1e-05
Stopping the search.


===== Final Ensemble =====
Selected models: [(42, 'rsf', 158), (42, 'rsf', 229), (42, 'gbsa', 236), (42, 'rsf', 175), (42, 'rsf', 86)]
Model counts: {(42, 'rsf', 158): 1, (42, 'rsf', 229): 1, (42, 'gbsa', 236): 1, (42, 'rsf', 175): 1, (42, 'rsf', 86): 1}
Model weights: {(42, 'rsf', 158): 0.2, (42, 'rsf', 229): 0.2, (42, 'gbsa', 236): 0.2, (42, 'rs

In [14]:
print(ensemble_result)

EnsembleModel(model_weights={(42, 'rsf', 158): 0.2, (42, 'rsf', 229): 0.2, (42, 'gbsa', 236): 0.2, (42, 'rsf', 175): 0.2, (42, 'rsf', 86): 0.2}, ensemble_score=None)


In [15]:
from Utils.ensemble_utils import search_ensemble_weight
ensemble_result = search_ensemble_weight(
    oof_result=full_oof_result, 
    model_dict=ensemble_result, 
    label=y_full, 
    eval_horizons=event_horizon
)


Weight Searching in [(42, 'rsf', 158), (42, 'rsf', 229), (42, 'gbsa', 236), (42, 'rsf', 175), (42, 'rsf', 86)]...


  0%|          | 0/100000 [00:00<?, ?it/s]

In [16]:
from Utils.utils import print_metric_score
print_metric_score(ensemble_result)

C-index:  0.9493209672076847
Mean Brier:  0.023402331158791003
Hybrid Score:  0.9684146583511517
